# Football RAG Evaluation

Notebook này chạy evaluate retrieval cho `bm25`, `vector`, `hybrid` và vẽ các biểu đồ trực quan. Chạy notebook từ thư mục gốc dự án `C:\\Users\\Admin\\Desktop\\football\\RAG` hoặc để cell đầu tự chuyển về project root.

In [ ]:
from pathlib import Path
import json
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "football_rag").exists():
    PROJECT_ROOT = Path(r"C:\Users\Admin\Desktop\football\RAG")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Notebook kernel Python:", sys.executable)
PROJECT_ROOT

## Kiểm tra dependencies trong đúng kernel

Nếu bạn đã chạy `pip install -r requirements-local.txt` ở PowerShell nhưng notebook vẫn báo thiếu package, nghĩa là Jupyter đang dùng Python kernel khác. Cell dưới đây cài package vào đúng kernel đang chạy notebook bằng `sys.executable -m pip`.

In [ ]:
import importlib.util
import subprocess

INSTALL_MISSING = True
required_modules = {
    "faiss": "faiss-cpu",
    "sentence_transformers": "sentence-transformers",
    "matplotlib": "matplotlib",
}
missing = [package for module, package in required_modules.items() if importlib.util.find_spec(module) is None]

if missing:
    print("Thiếu package trong kernel hiện tại:", missing)
    print("Kernel Python:", sys.executable)
    if INSTALL_MISSING:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements-local.txt"])
        print("Đã cài xong. Nếu import vẫn lỗi, hãy restart kernel rồi chạy lại notebook.")
    else:
        print("Chạy lệnh này trong terminal:")
        print(f'"{sys.executable}" -m pip install -r requirements-local.txt')
else:
    print("Dependencies OK trong kernel hiện tại:", sys.executable)

## Cấu hình evaluate

In [ ]:
from football_rag.evaluate import DEFAULT_CASES, find_hit_rank
from football_rag.retrieve import hybrid_search

BM25_INDEX = Path("output/retrieval/bm25_index.json")
VECTOR_DIR = Path("output/retrieval/vector-local")
TOP_K = 5
MODES = ["bm25", "vector", "hybrid"]

assert BM25_INDEX.exists(), f"Missing {BM25_INDEX}. Chạy: python -m football_rag.bm25 build"
assert (VECTOR_DIR / "faiss.index").exists(), f"Missing vector index. Chạy: python -m football_rag.vector build"

DEFAULT_CASES

## Chạy evaluate cho từng mode

In [ ]:
rows = []
raw_results = {}

for mode in MODES:
    raw_results[mode] = {}
    for case in DEFAULT_CASES:
        results = hybrid_search(
            case["question"],
            bm25_index_path=BM25_INDEX,
            vector_dir=VECTOR_DIR,
            mode=mode,
            top_k=TOP_K,
        )
        raw_results[mode][case["id"]] = results
        rank = find_hit_rank(results, case["expected_contains"])
        rows.append(
            {
                "mode": mode,
                "case_id": case["id"],
                "question": case["question"],
                "expected_contains": ", ".join(case["expected_contains"]),
                "hit": rank is not None,
                "rank": rank,
                "reciprocal_rank": 1 / rank if rank else 0.0,
                "top_1_chunk": results[0].get("chunk_id") if results else None,
                "top_1_doc_type": results[0].get("doc_type") if results else None,
                "top_chunks": [item.get("chunk_id") for item in results],
            }
        )

rows[:2]

## Bảng kết quả

In [ ]:
try:
    import pandas as pd
    df = pd.DataFrame(rows)
    display(df[["mode", "case_id", "hit", "rank", "reciprocal_rank", "top_1_chunk", "top_1_doc_type"]])
except ImportError:
    df = None
    for row in rows:
        print(row)

## Tổng hợp metric

In [ ]:
summary = []
for mode in MODES:
    mode_rows = [row for row in rows if row["mode"] == mode]
    summary.append(
        {
            "mode": mode,
            "case_count": len(mode_rows),
            "hit_at_k": sum(1 for row in mode_rows if row["hit"]) / len(mode_rows),
            "mrr": sum(row["reciprocal_rank"] for row in mode_rows) / len(mode_rows),
        }
    )

if df is not None:
    display(pd.DataFrame(summary))
else:
    print(json.dumps(summary, ensure_ascii=False, indent=2))

## Biểu đồ Hit@K và MRR theo mode

In [ ]:
import matplotlib.pyplot as plt

modes = [item["mode"] for item in summary]
hit_values = [item["hit_at_k"] for item in summary]
mrr_values = [item["mrr"] for item in summary]

fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
axes[0].bar(modes, hit_values, color=["#64748b", "#0f766e", "#2563eb"])
axes[0].set_title(f"Hit@{TOP_K} theo mode")
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel("Hit@K")

axes[1].bar(modes, mrr_values, color=["#64748b", "#0f766e", "#2563eb"])
axes[1].set_title("MRR theo mode")
axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel("MRR")

plt.show()

## Biểu đồ rank theo từng câu hỏi

In [ ]:
case_ids = [case["id"] for case in DEFAULT_CASES]
x = range(len(case_ids))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 5), constrained_layout=True)
for offset, mode in enumerate(MODES):
    mode_rows = [row for row in rows if row["mode"] == mode]
    ranks = [row["rank"] if row["rank"] is not None else TOP_K + 1 for row in mode_rows]
    positions = [i + (offset - 1) * width for i in x]
    ax.bar(positions, ranks, width=width, label=mode)

ax.set_title("Rank của kết quả đúng theo từng câu hỏi")
ax.set_ylabel(f"Rank, {TOP_K + 1} = miss")
ax.set_xticks(list(x))
ax.set_xticklabels(case_ids, rotation=25, ha="right")
ax.invert_yaxis()
ax.legend()
plt.show()

## Biểu đồ loại tài liệu top-1

In [ ]:
from collections import Counter

fig, axes = plt.subplots(1, len(MODES), figsize=(14, 4), constrained_layout=True)
for ax, mode in zip(axes, MODES):
    mode_rows = [row for row in rows if row["mode"] == mode]
    counts = Counter(row["top_1_doc_type"] or "unknown" for row in mode_rows)
    labels = list(counts.keys())
    values = list(counts.values())
    ax.pie(values, labels=labels, autopct="%1.0f%%", startangle=90)
    ax.set_title(f"Top-1 doc_type: {mode}")

plt.show()

## Lưu report JSON

In [ ]:
report = {
    "top_k": TOP_K,
    "summary": summary,
    "rows": rows,
}

output_path = Path("output/evaluation/notebook_eval_report.json")
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
output_path